In [ ]:
!pip install -U ultralytics opencv-python


In [ ]:
!pip install RPi.GPIO


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
from ultralytics import YOLOWorld
import cv2
import time
import matplotlib.pyplot as plt

BUZZER_AVAILABLE = False  # RPi.GPIO can be used. Hardware for buzzer alarm can be integrated.

model = YOLOWorld("/content/drive/MyDrive/yolov8s_worldv2/runs/weights/last.pt")
cap = cv2.VideoCapture("/content/drive/MyDrive/vid_testing/pedestrians_alertSys_1.mp4")


#Setting desired restricted zone in the format (x1, y1, x2, y2)- top left, bottom right
RESTRICTED_ZONE = (200, 250, 300, 350)

#initial frame with restricted zone
ret, frame = cap.read()
if ret:
    temp = frame.copy()
    cv2.rectangle(temp, (RESTRICTED_ZONE[0], RESTRICTED_ZONE[1]), (RESTRICTED_ZONE[2], RESTRICTED_ZONE[3]), (0, 0, 255), 2)
    plt.figure(figsize=(8,6))
    plt.imshow(cv2.cvtColor(temp, cv2.COLOR_BGR2RGB))
    plt.title('Restricted Zone Preview (Red Box)')
    plt.axis('off')
    plt.show()


In [ ]:
# Detection and Inline Display
try:
    max_frames = 100  # limit frames in Colab, remove or increase for larger loop
    for i in range(max_frames):
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame)

        alarm_triggered = False
        for box in results[0].boxes:
          cls_id = int(box.cls)
          if cls_id == 0:  # 'person'
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Draw the bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label = f"Person {box.conf[0]:.2f}"
            cv2.putText(frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

            # Check if person is inside restricted zone
            rx1, ry1, rx2, ry2 = RESTRICTED_ZONE
            if x1 > rx1 and y1 > ry1 and x2 < rx2 and y2 < ry2:
              alarm_triggered = True


        # Draw the restricted zone
        cv2.rectangle(frame, (RESTRICTED_ZONE[0], RESTRICTED_ZONE[1]), (RESTRICTED_ZONE[2], RESTRICTED_ZONE[3]), (0, 0, 255), 2)

        # Alarm logic
        if alarm_triggered:
          print("Person inside restricted zone! Alarm Triggered.")
          if BUZZER_AVAILABLE:
            pass
          time.sleep(1)

        else:
            if BUZZER_AVAILABLE:
                pass

        # Inline image display
        plt.figure(figsize=(8,6))
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.title(f'Detection Output - Frame {i+1}')
        plt.axis('off')
        plt.show()

except KeyboardInterrupt:
    print("Stopped by user.")
finally:
    cap.release()